# Step 1: Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Step 2: Create Sample Dataset

In [3]:
# The dataset represents the relationship:
# y = 2x + 1
X = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0]
])
 
y = torch.tensor([
    [3.0],
    [5.0],
    [7.0],
    [9.0],
    [11.0]
])

# Step 3: Build the Neural Network

In [4]:
class SimpleRegressionModel(nn.Module):
 
    def __init__(self):
        super().__init__()
 
        self.linear = nn.Linear(
            in_features=1,
            out_features=1
        )
 
    def forward(self, x):
        return self.linear(x)



In [5]:
# Create the model:
model = SimpleRegressionModel()
print(model)

SimpleRegressionModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)


# Step 4: Configure Training

In [6]:
criterion = nn.MSELoss()
 
optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Step 5: Train the Model

In [8]:
epochs = 1000
 
for epoch in range(epochs):
 
    predictions = model(X)
 
    loss = criterion(
        predictions,
        y
    )
 
    optimizer.zero_grad()
 
    loss.backward()
 
    optimizer.step()
 
print("Training Complete")

Training Complete


# Step 6: Test the Model

In [9]:
test_input = torch.tensor([[10.0]])
 
prediction = model(test_input)
 
print(
    "Prediction:",
    prediction.item()
)

Prediction: 21.00235366821289


# Step 7: Save the Model

In [11]:
torch.save(
    model.state_dict(),
    "simple_model.pth"
)
print("Model Saved")

Model Saved


# Step 8: Load the Model

In [13]:
loaded_model = SimpleRegressionModel()
 
loaded_model.load_state_dict(
    torch.load("simple_model.pth")
)
 
loaded_model.eval()
 
print("Model Loaded")


Model Loaded


# Step 9: Perform Inference

In [14]:
sample = torch.tensor([[20.0]])
 
with torch.no_grad():
 
    prediction = loaded_model(sample)
 
print(
    "Prediction:",
    prediction.item()
)


Prediction: 41.00603485107422


# Step 10: Create a REST API

### Install FastAPI: pip install fastapi uvicorn
### Create a file named: app.py


In [17]:
# Add the following code
from fastapi import FastAPI
import torch
import torch.nn as nn
 
app = FastAPI()
 
class SimpleRegressionModel(nn.Module):
 
    def __init__(self):
        super().__init__()
 
        self.linear = nn.Linear(
            1,
            1
        )
 
    def forward(self, x):
        return self.linear(x)
 
model = SimpleRegressionModel()
 
model.load_state_dict(
    torch.load("simple_model.pth")
)
 
model.eval()
 
@app.get("/predict/{value}")
 
def predict(value: float):
 
    x = torch.tensor(
        [[value]],
        dtype=torch.float32
    )
 
    with torch.no_grad():
        prediction = model(x)
 
    return {
        "input": value,
        "prediction": prediction.item()
    }
 


# Step 11: Start the API Server

Open a terminal and run: uvicorn app:app --reload <br>
Expected output:Uvicorn running on http://127.0.0.1:8000


# Step 12: Test the API

Open a browser: http://127.0.0.1:8000/predict/15